# Driving Narrator - YOLOv11 Training v3

**Purpose:** Train YOLOv11 Nano on LISA Traffic Signs with proper stratified split.

**Features:**
- Auto-resume if interrupted (shows which epoch)
- Only `last.pt` and `best.pt` saved (no clutter)
- Exports to OpenVINO INT8

**Output:** `/content/drive/MyDrive/Driving_Narrator/DN_Training_v3/`

## 1. Setup Environment

In [5]:
!gdown "https://drive.google.com/uc?id=1Aj3SrE08AwKaJK9PYEH0IXp1s_iwl0Tb" --fuzzy -O /content/dataset.zip
!unzip -q /content/dataset.zip -d /content/
!rm /content/dataset.zip

Downloading...
From (original): https://drive.google.com/uc?id=1Aj3SrE08AwKaJK9PYEH0IXp1s_iwl0Tb
From (redirected): https://drive.google.com/uc?id=1Aj3SrE08AwKaJK9PYEH0IXp1s_iwl0Tb&confirm=t&uuid=a0eb74c3-48f1-465b-b388-c853863df369
To: /content/dataset.zip
100% 1.10G/1.10G [00:16<00:00, 66.0MB/s]


In [5]:
# Connect to Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Setup YOLOv11 and OpenVINO environment
!pip install ultralytics openvino-dev nncf onnxscript -q

# Check GPU
!nvidia-smi

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/bin/bash: line 1: nvidia-smi: command not found


In [6]:
# Define training artifacts and checkpoint paths
import os
from pathlib import Path

# Paths
DRIVE_BASE = Path("/content/drive/MyDrive/Driving_Narrator")
TRAINING_DIR = DRIVE_BASE / "DN_Training_v3"  # Output folder
CHECKPOINT_DIR = TRAINING_DIR / "checkpoints"
EXPORT_DIR = TRAINING_DIR / "exports"

# Create directories
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Training outputs: {TRAINING_DIR}")
print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Exports: {EXPORT_DIR}")

Training outputs: /content/drive/MyDrive/Driving_Narrator/DN_Training_v3
Checkpoints: /content/drive/MyDrive/Driving_Narrator/DN_Training_v3/checkpoints
Exports: /content/drive/MyDrive/Driving_Narrator/DN_Training_v3/exports


## 2. Load Dataset (to /content for speed)

Dataset is copied from Drive to Colab's local SSD for faster training.

In [4]:
# Copy dataset from Drive to /content (local SSD = faster I/O)
import shutil

DATASET_DRIVE = DRIVE_BASE / "LISA_stratified"
DATASET_LOCAL = Path("/content/LISA_stratified")

if DATASET_LOCAL.exists():
    print("Dataset already in /content, skipping copy")
else:
    print("Copying dataset from Drive to /content (faster training)...")
    shutil.copytree(DATASET_DRIVE, DATASET_LOCAL)
    print("✅ Done!")

# Verify
!ls -la /content/LISA_stratified/
!echo "Train images:" && ls /content/LISA_stratified/train/images | wc -l
!echo "Valid images:" && ls /content/LISA_stratified/valid/images | wc -l
!echo "Test images:" && ls /content/LISA_stratified/test/images | wc -l

Dataset already in /content, skipping copy
total 24
drwxr-xr-x 5 root root 4096 Dec 17 17:47 .
drwxr-xr-x 1 root root 4096 Dec 17 17:48 ..
-rw-r--r-- 1 root root 1009 Dec 17 16:37 data.yaml
drwxr-xr-x 4 root root 4096 Dec 17 17:47 test
drwxr-xr-x 4 root root 4096 Dec 17 16:35 train
drwxr-xr-x 4 root root 4096 Dec 17 16:35 valid
Train images:
11089
Valid images:
3159
Test images:
1629


In [5]:
# Fix data.yaml paths for Colab
data_yaml_content = '''train: /content/LISA_stratified/train/images
val: /content/LISA_stratified/valid/images
test: /content/LISA_stratified/test/images

nc: 47
names: ['addedLane', 'curveLeft', 'curveRight', 'dip', 'doNotEnter', 'doNotPass', 'intersection', 'keepRight', 'laneEnds', 'merge', 'noLeftTurn', 'noRightTurn', 'pedestrianCrossing', 'rampSpeedAdvisory20', 'rampSpeedAdvisory35', 'rampSpeedAdvisory40', 'rampSpeedAdvisory45', 'rampSpeedAdvisory50', 'rampSpeedAdvisoryUrdbl', 'rightLaneMustTurn', 'roundabout', 'school', 'schoolSpeedLimit25', 'signalAhead', 'slow', 'speedLimit15', 'speedLimit25', 'speedLimit30', 'speedLimit35', 'speedLimit40', 'speedLimit45', 'speedLimit50', 'speedLimit55', 'speedLimit65', 'speedLimitUrdbl', 'stop', 'stopAhead', 'thruMergeLeft', 'thruMergeRight', 'thruTrafficMergeLeft', 'truckSpeedLimit55', 'turnLeft', 'turnRight', 'yield', 'yieldAhead', 'zoneAhead25', 'zoneAhead45']
'''

with open('/content/LISA_stratified/data.yaml', 'w') as f:
    f.write(data_yaml_content)

print("✅ data.yaml updated with Colab paths")

✅ data.yaml updated with Colab paths


## 3. Training with Auto-Resume

**Checkpoint behavior:**
- `last.pt` = overwritten after each epoch (for resume)
- `best.pt` = overwritten when mAP improves
- No extra `epoch_N.pt` files (clean folder)

In [6]:
from ultralytics import YOLO
import torch

# Optimized training configuration for T4 GPU
CONFIG = {
    'epochs': 50,
    'batch': 64,            # Increased: better GPU utilization
    'imgsz': 640,
    'patience': 15,
    'workers': 4,           # Parallel data loading
    'amp': True,            # Mixed precision: faster + less memory
    'project': str(CHECKPOINT_DIR),
    'name': 'yolo11n_lisa',
    'exist_ok': True,
    'save': True,
    'save_period': -1,      # Only last.pt and best.pt
    'plots': True,
    'verbose': True,
}

# Paths
checkpoint_path = CHECKPOINT_DIR / 'yolo11n_lisa' / 'weights' / 'last.pt'
best_path = CHECKPOINT_DIR / 'yolo11n_lisa' / 'weights' / 'best.pt'

if checkpoint_path.exists():
    ckpt = torch.load(str(checkpoint_path), map_location='cpu')
    last_epoch = ckpt.get('epoch', 0)
    print(f'🔄 Found checkpoint: {checkpoint_path}')
    print(f'   Last completed epoch: {last_epoch}')
    print(f'   Resuming from epoch {last_epoch + 1}...')
    del ckpt
    model = YOLO(str(checkpoint_path))
    results = model.train(
        data='/content/LISA_stratified/data.yaml',
        resume=True,
        **CONFIG
    )
else:
    print('🆕 Starting fresh training from epoch 1...')
    model = YOLO('yolo11n.pt')
    results = model.train(
        data='/content/LISA_stratified/data.yaml',
        **CONFIG
    )

print('\n✅ Training complete!')
print(f'Best model: {best_path}')

🆕 Starting fresh training from epoch 1...
Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/LISA_stratified/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11n_lisa, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=T

## 4. Evaluate on Test Set

In [7]:
from ultralytics import YOLO
import torch
checkpoint_path = CHECKPOINT_DIR / 'yolo11n_lisa' / 'weights' / 'last.pt'
best_path = CHECKPOINT_DIR / 'yolo11n_lisa' / 'weights' / 'best.pt'
best_model = YOLO(str(best_path))


In [6]:
# Evaluate best model on test set
best_model = YOLO(str(best_path))

print("Evaluating on TEST set...")
metrics = best_model.val(
    data='/content/LISA_stratified/data.yaml',
    split='test',
    imgsz=640,
)

print("\n=== TEST SET RESULTS ===")
print(f"mAP@0.5:      {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:    {metrics.box.mp:.4f}")
print(f"Recall:       {metrics.box.mr:.4f}")

Evaluating on TEST set...
Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11n summary (fused): 100 layers, 2,591,317 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1015.5±523.7 MB/s, size: 74.9 KB)
val: Scanning /content/LISA_stratified/test/labels... 1629 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1629/1629 1.8Kit/s 0.9s
val: /content/LISA_stratified/test/images/pedestrianCrossing_1333397756-avi_image29_png.rf.8699643b9c2b9af49f635e567a06c9e4.jpg: 1 duplicate labels removed
val: New cache created: /content/LISA_stratified/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 102/102 5.4s/it 9:07
                   all       1629       1903      0.957      0.936      0.969       0.82
             addedLane         74         74      0.998      0.959      0.991      0.852
             curveLeft          8          8      0.965   

## 5. Export All Formats

In [11]:
!pip install onnx onnxslim -q

# Then run your export cell
from ultralytics import YOLO
best_model = YOLO('/content/drive/MyDrive/Driving_Narrator/DN_Training_v3/checkpoints/yolo11n_lisa/weights/best.pt')

# Export to ONNX
print("Exporting to ONNX...")
onnx_path = best_model.export(format='onnx', imgsz=416, simplify=True)
print(f"✅ ONNX: {onnx_path}")

# Export to OpenVINO FP32
print("\nExporting to OpenVINO FP32...")
fp32_path = best_model.export(format='openvino', imgsz=416, half=False)
print(f"✅ FP32: {fp32_path}")

# INT8 already done - skip to avoid re-running
print("\n✅ INT8: Already exported")

## 6. Save Plots for Paper

In [12]:
import shutil

RESULTS_DIR = CHECKPOINT_DIR / 'yolo11n_lisa'

plots_to_save = [
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'results.png',
    'labels.jpg',
    'labels_correlogram.jpg',
    'F1_curve.png',
    'P_curve.png',
    'R_curve.png',
    'PR_curve.png',
]

print("Saving plots...")
for plot in plots_to_save:
    src = RESULTS_DIR / plot
    if src.exists():
        shutil.copy(src, EXPORT_DIR / plot)
        print(f"  ✅ {plot}")

print(f"\n📊 Plots saved to: {EXPORT_DIR}")

Saving plots...
  ✅ confusion_matrix.png
  ✅ confusion_matrix_normalized.png
  ✅ results.png
  ✅ labels.jpg

📊 Plots saved to: /content/drive/MyDrive/Driving_Narrator/DN_Training_v3/exports


## 7. Save Per-Class Metrics

In [14]:
from ultralytics import YOLO

best_model = YOLO('/content/drive/MyDrive/Driving_Narrator/DN_Training_v3/checkpoints/yolo11n_lisa/weights/best.pt')

# Re-run evaluation to get metrics
metrics = best_model.val(
    data='/content/LISA_stratified/data.yaml',
    split='test',
    imgsz=640,
    plots=True
)

print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")

Ultralytics 8.3.40 🚀 Python-3.12.12 torch-2.9.1+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11n summary (fused): 238 layers, 2,591,317 parameters, 0 gradients, 6.4 GFLOPs


val: Scanning /content/LISA_stratified/test/labels.cache... 1629 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1629/1629 [00:00<?, ?it/s]

val: /content/LISA_stratified/test/images/pedestrianCrossing_1333397756-avi_image29_png.rf.8699643b9c2b9af49f635e567a06c9e4.jpg: 1 duplicate labels removed



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 102/102 [10:06<00:00,  5.95s/it]


                   all       1629       1903      0.957      0.936      0.969      0.821
             addedLane         74         74      0.998      0.959      0.991      0.852
             curveLeft          8          8      0.965          1      0.995      0.899
            curveRight         13         13      0.999          1      0.995      0.839
                   dip          9          9          1      0.928      0.995      0.755
            doNotEnter          7          7      0.972          1      0.995      0.664
             doNotPass          3          3      0.915          1      0.995      0.842
          intersection          1          1          1          0      0.995      0.796
             keepRight         77         77      0.932          1      0.983      0.817
              laneEnds         47         49          1       0.97      0.995      0.853
                 merge         64         64      0.996          1      0.995      0.876
            noLeftTur

In [15]:
import json

class_names = best_model.names
results_dict = {
    'overall': {
        'mAP50': float(metrics.box.map50),
        'mAP50_95': float(metrics.box.map),
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
    },
    'per_class': {}
}

if hasattr(metrics.box, 'ap50') and metrics.box.ap50 is not None:
    for i, ap in enumerate(metrics.box.ap50):
        if i < len(class_names):
            results_dict['per_class'][class_names[i]] = round(float(ap), 4)

with open(EXPORT_DIR / 'test_metrics.json', 'w') as f:
    json.dump(results_dict, f, indent=2)

print(f"✅ Metrics saved to: {EXPORT_DIR / 'test_metrics.json'}")

✅ Metrics saved to: /content/drive/MyDrive/Driving_Narrator/DN_Training_v3/exports/test_metrics.json


## 8. Copy All Models to Drive

In [ ]:
import shutil
import os

print("Copying models to Drive...")

# best.pt
shutil.copy(best_path, EXPORT_DIR / 'best.pt')
print("  ✅ best.pt")

# ONNX
onnx_file = Path(str(best_path).replace('.pt', '.onnx'))
if onnx_file.exists():
    shutil.copy(onnx_file, EXPORT_DIR / 'best.onnx')
    print("  ✅ best.onnx")

# OpenVINO FP32
fp32_folder = Path(str(best_path).replace('.pt', '_openvino_model'))
if fp32_folder.exists():
    shutil.copytree(fp32_folder, EXPORT_DIR / 'best_openvino_model', dirs_exist_ok=True)
    print("  ✅ best_openvino_model/")

# OpenVINO INT8
int8_folder = Path(str(best_path).replace('.pt', '_openvino_int8_model'))
if int8_folder.exists():
    shutil.copytree(int8_folder, EXPORT_DIR / 'best_int8_openvino_model', dirs_exist_ok=True)
    print("  ✅ best_int8_openvino_model/")

print(f"\n📦 All exports: {EXPORT_DIR}")
!ls -la {EXPORT_DIR}

## 9. Summary for Paper

In [16]:
def get_size(path):
    import os
    if os.path.isfile(path):
        return os.path.getsize(path) / (1024*1024)
    elif os.path.isdir(path):
        total = sum(os.path.getsize(os.path.join(dp, f)) for dp, dn, fn in os.walk(path) for f in fn)
        return total / (1024*1024)
    return 0

print("=" * 60)
print("MODEL SUMMARY FOR PAPER")
print("=" * 60)
print(f"{'Model':<30} {'Size (MB)':<15} {'Format'}")
print("-" * 60)

models = [
    ('best.pt', 'PyTorch'),
    ('best.onnx', 'ONNX'),
    ('best_openvino_model', 'OpenVINO FP32'),
    ('best_int8_openvino_model', 'OpenVINO INT8'),
]

for name, fmt in models:
    path = EXPORT_DIR / name
    if path.exists():
        size = get_size(path)
        print(f"{name:<30} {size:<15.2f} {fmt}")

print("=" * 60)
print(f"\nTest mAP@0.5: {results_dict['overall']['mAP50']:.2%}")
print(f"Test mAP@0.5:0.95: {results_dict['overall']['mAP50_95']:.2%}")
print(f"Precision: {results_dict['overall']['precision']:.2%}")
print(f"Recall: {results_dict['overall']['recall']:.2%}")

MODEL SUMMARY FOR PAPER
Model                          Size (MB)       Format
------------------------------------------------------------

Test mAP@0.5: 96.86%
Test mAP@0.5:0.95: 82.14%
Precision: 95.72%
Recall: 93.64%


## 10. Summary

In [17]:
print(f"""
╔══════════════════════════════════════════════════════╗
║              TRAINING COMPLETE!                      ║
╠══════════════════════════════════════════════════════╣
║ Models saved to:                                     ║
║   {EXPORT_DIR}
║                                                      ║
║ Files:                                               ║
║   - best.pt                                          ║
║   - best_int8_openvino_model/                        ║
║                                                      ║
║ TEST SET mAP@0.5: {metrics.box.map50:.2%}                          ║
╚══════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════╗
║              TRAINING COMPLETE!                      ║
╠══════════════════════════════════════════════════════╣
║ Models saved to:                                     ║
║   /content/drive/MyDrive/Driving_Narrator/DN_Training_v3/exports                                       
║                                                      ║
║ Files:                                               ║
║   - best.pt                                          ║
║   - best_int8_openvino_model/                        ║
║                                                      ║
║ TEST SET mAP@0.5: 96.86%                          ║
╚══════════════════════════════════════════════════════╝

